In [ ]:
# ===========================================
# سیستم RAG فارسی با پرامپت اصولی و جامع
# تمرکز بر اصول کلی به جای مثال‌های خاص
# ===========================================

!pip install -q sentence-transformers faiss-cpu requests pandas tqdm scikit-learn

import os
import time
import re
import json
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# -------- تنظیمات API --------
API_KEY = ""  # کلید OpenRouter API خود را اینجا جایگزین کنید
API_URL = "https://openrouter.ai/api/v1/chat/completions"
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# -------- بارگذاری مدل embedding فارسی --------
print("🔧 بارگذاری مدل embedding فارسی...")
embedder = SentenceTransformer('xmanii/maux-gte-persian', trust_remote_code=True)

# -------- دیتاست آموزشی متعادل (بدون bias خاص) --------
examples = {
    "text": [
        # === نفرتآمیز قومی (40 مثال) ===
        "مرگ بر ترک‌های نفهم",
        "عرب کثیف",
        "کردها همه شورشگرن",
        "بلوچ‌ها قابل اعتماد نیستن",
        "فارس‌ها خودشونو برتر می‌بینن",
        "آذری‌ها زیاد پرحرفن",
        "گیلک‌ها عقب‌افتادن",
        "مازنی‌ها مشکل‌سازن",
        "سیستانی‌ها معتادن",
        "قشقایی‌ها بی‌تمدنن",
        "لر ضعیف و نامرد",
        "کاشکایی‌ها دعواجون",
        "ایلامی‌ها نباید رای داشته باشن",
        "خراسانی‌ها سرد و بی‌احساسن",
        "اردبیلی‌ها عبوس و بدقلقن",
        "کیرم تو ترک‌ها",
        "گه بخورن کردها",
        "کوسشعر این عرب‌ها",
        "ترک‌ها همیشه خیانت می‌کنن",
        "کردها غیرقابل اعتمادن",
        "آذری‌ها مخربن",
        "بلوچ‌ها مشکوک هستن",
        "گیلک‌ها شلوغ و مزاحمن",
        "مازنی‌ها دردسرن",
        "سیستانی‌ها خطرناکن",
        "قشقایی‌ها وحشین",
        "فارس‌ها مغرورن",
        "لرها توی شهر مشکل دارن",
        "ترک‌ها دیر میان همیشه",
        "آذری‌ها پرحرف هستن زیادی",
        "کردها عاشق دعوان",
        "گیلک‌ها خیلی شلوغن",
        "با این لهجه معلومه اهل کجاست",
        "اسمش میگه چه قومیه",
        "این قوم که میاد محل خراب میشه",
        "رسوماشون عجیبه",
        "غذاهاشون بوی بد میده",
        "نژادشون معلومه از رفتارشون",
        "این قبیله مشکل‌سازه",
        "عاداتشون قابل تحمل نیست",

        # === نفرتآمیز ملیتی (40 مثال) ===
        "کیرم تو این ایران",
        "افغان‌ها همه دزدن",
        "پاکستانی‌ها بی‌فرهنگن",
        "عراقی‌ها جنگجون",
        "چینی‌ها دروغگون",
        "هندی‌ها کثیفن",
        "لبنانی‌ها دلالن",
        "سوری‌ها مشکل‌سازن",
        "ترکیه‌ای‌ها خائنن",
        "روسی‌ها مستن",
        "آلمانی‌ها سردن",
        "یمنی‌ها عقب‌افتادن",
        "فلسطینی‌ها دردسرن",
        "کیرم تو افغان‌ها",
        "گه بخورن پاکستانی‌ها",
        "کوسشعر چینی‌ها",
        "افغان‌ها نابود شن",
        "پاکستانی‌ها اخراج شن",
        "عراقی‌ها امنیت رو خراب می‌کنن",
        "چینی‌ها ویروس می‌آرن",
        "هندی‌ها بیماری دارن",
        "لبنانی‌ها منافع‌طلبن",
        "سوری‌ها کشور رو ناامن کردن",
        "ترکیه‌ای‌ها دشمن تاریخین",
        "روسی‌ها اشغالگرن",
        "آلمانی‌ها نژادپرستن",
        "خارجی‌ها مشکل‌سازن",
        "اجنبی‌ها فرهنگ رو خراب می‌کنن",
        "غیربومی‌ها حق زندگی ندارن",
        "این که از اون ور مرزه مشکوکه",
        "ملیت خارجی داره خطرناکه",
        "اجانب همیشه مشکل دارن",
        "بیگانه‌ها آرامش رو از بین می‌برن",
        "افغان‌ها چای قوی می‌خورن زیادی",
        "پاکستانی‌ها پرحرفن",
        "چینی‌ها کوتاه قدن",
        "هندی‌ها ادویه زیاد می‌ریزن",
        "خارجی که هست فرق داره",
        "فرهنگشون با ما نمیخونه",
        "زبونشون عجیبه",

        # === نفرتآمیز مذهبی (20 مثال) ===
        "بهایی‌ها خائنن",
        "سنی‌ها تروریستن",
        "مسیحی‌ها کافرن",
        "آتئیست‌ها فاسدن",
        "شیعه‌ها ریاکارن",
        "یهودی‌ها حریصن",
        "زرتشتی‌ها خرافاتین",
        "سلفی‌ها خطرناکن",
        "درویش‌ها انگلن",
        "وهابی‌ها خشونت‌طلبن",
        "کلیسا مشکل‌سازه",
        "مذاهب غیر اسلامی باطلن",
        "صوفیان دین رو منحرف می‌کنن",
        "غیرمسلمان‌ها دشمنن",
        "کافرها نابود شن",
        "فقط شیعه مسلمان واقعیه",
        "اهل سنت پتانسیل تروریستی دارن",
        "کلیساها مرکز فسادن",
        "آتئیست‌ها خطر جامعه‌ان",
        "مذهب‌های دیگه مشکوکن",

        # === نفرتآمیز جنسیتی (20 مثال) ===
        "زن نباید رییس باشه",
        "دخترها احمقن",
        "همجنسگراها بیمارن",
        "فمینیست‌ها خانواده رو نابود می‌کنن",
        "زن‌ها فقط دنبال پولن",
        "زنای محجبه عقب‌افتادن",
        "زن‌های بی‌حجاب فاسدن",
        "ترنس‌ها مریضن",
        "زن‌ها شایسته کار نیستن",
        "پسرها دروغگون",
        "زن باید مطیع باشه",
        "زن‌های مطلقه مشکل دارن",
        "مردهای خونه‌دار ضعیفن",
        "اقلیت‌های جنسی درمان شن",
        "همجنسگرایی حرامه",
        "زن‌ها عقل کامل ندارن",
        "پسرها فاسدن",
        "زن مجرد بدبخته",
        "زن‌های جوان سطحین",
        "مردان برترن",

        # === نفرتآمیز سیاسی (15 مثال) ===
        "اصلاح‌طلب‌ها خائنن",
        "اصولگراها متحجرن",
        "ساندیسی دشمن فرهنگ",
        "کمونیست‌ها بی‌خدان",
        "ناسیونالیست‌ها فاشیستن",
        "سلطنت‌طلب‌ها وابسته‌ان",
        "چپ‌ها آشوبگرن",
        "اخوند عقب‌افتادس",
        "دموکرات‌ها ریاکارن",
        "جمهوری‌خواهان ظالمن",
        "سوسیالیست‌ها رویاپردازن",
        "آشوبگران مخربن",
        "اقازاده‌ها دشمن مردمن",
        "سیاستمداران فاسدن",
        "بسیجیا خرابکارن",

        # === نفرتآمیز سنی (15 مثال) ===
        "پیرها بار اقتصادین",
        "سالمندها خرفتن",
        "پیرمردها قابلیت ندارن",
        "زنای پیر فضولن",
        "پیرها نباید تصمیم بگیرن",
        "سالمندها کنار برن",
        "پیرها پول هدر می‌دن",
        "این بابای پیر چی می‌فهمه",
        "بچه‌ها نابالغن",
        "جوونای امروزی بی‌ادبن",
        "نسل جدید فقط عیش می‌خواد",
        "بچه‌های الان مسئولیت ندارن",
        "جوونا تند و عجولن",
        "نوجوونا احترام نمی‌ذارن",
        "بچه‌ها چی می‌فهمن",

        # === توهین‌آمیز شخصی (20 مثال) ===
        "خفه شو احمق",
        "جاکش بی‌ادب",
        "گاو نفهم",
        "بی‌مغز مطلق",
        "کوسکش عوضی",
        "خری هستی",
        "گه خوری تو رو کرد",
        "زن خودش جندست",
        "دیوونه",
        "بی‌عرضه",
        "بی‌شعور لعنتی",
        "کله‌پوک اینجا رو ترک نکن",
        "برو گمشو",
        "کیرم تو خودت",
        "کس ننت",
        "گه بخور",
        "کوسشعر می‌گی",
        "به کیرم",
        "بی‌فرهنگ",
        "خرابکار",

        # === عادی مثبت (20 مثال) ===
        "سلام چطوری؟",
        "دمت گرم داداش",
        "چه فیلم خوبی بود",
        "خوشحالم آشنا شدیم",
        "سلامت باشی",
        "کار عالی بود",
        "موفق باشی",
        "ممنون از همراهی",
        "بهترین دوستی",
        "کتاب خوبی بود",
        "غذا خوشمزه بود",
        "خوشحالم کنارتم",
        "سربلند باشی",
        "روز خوب باشه",
        "سلامتی برات",
        "شاد باشی",
        "خدا نگهدارت",
        "ممنون از کمک",
        "بهترین‌ها برات",
        "دستت درد نکنه",

        # === عادی خنثی (30 مثال) ===
        "هوا امروز خوب بود",
        "فردا جلسه داریم",
        "ترافیک زیاد بود",
        "قیمت بنزین بالا رفت",
        "کتابخانه تعطیل بود",
        "امتحان هفته آینده",
        "برق قطع بود",
        "هوا سرده",
        "محصول خوبه",
        "قیمت‌ها تغییر کرد",
        "کار خاصی ندارم",
        "تعطیلی اعلام شد",
        "ترافیک کم شده",
        "اکثر فروشگاه‌ها بازه",
        "ساعت کاری تغییر کرد",
        "به سلامتی سه کس سرباز",
        "تهران شهر بزرگیه با اقوام مختلف",
        "این شهر تنوع فرهنگی داره",
        "مردم مختلف اینجا زندگی می‌کنن",
        "هر قومی رسوم خاص داره",
        "افراد از کشورهای مختلف کار می‌کنن",
        "این محله ترکیب خوبی از فرهنگ‌هاست",
        "خواهرت خیلی باهوشه",
        "دوستم از استان دیگه‌ست",
        "همکارم اهل شیراز هست",
        "استاد ما ترک هست",
        "پزشکمون کرد هست",
        "مهندس پروژه افغانی هست",
        "ایران کشور چندقومیه",
        "فرهنگ‌های مختلف زیبان"
    ],
    "label": (
        ["تنفرآمیز"]*40 + ["تنفرآمیز"]*40 + ["تنفرآمیز"]*20 + ["تنفرآمیز"]*20 + ["تنفرآمیز"]*15 + ["تنفرآمیز"]*15 +
        ["توهین‌آمیز"]*20 +
        ["عادی"]*20 + ["عادی"]*30
    ),
    "hate_type": (
        ["قومی"]*40 + ["ملیتی"]*40 + ["مذهبی"]*20 + ["جنسیتی"]*20 + ["سیاسی"]*15 + ["سنی"]*15 +
        ["N/A"]*20 +
        ["N/A"]*20 + ["N/A"]*30
    ),
    "reason": (
        ["نفرت قومی"]*40 +
        ["نفرت ملیتی"]*40 +
        ["نفرت مذهبی"]*20 +
        ["نفرت جنسیتی"]*20 +
        ["نفرت سیاسی"]*15 +
        ["نفرت سنی"]*15 +
        ["توهین شخصی"]*20 +
        ["تعامل مثبت"]*20 +
        ["بیان عادی"]*30
    )
}

# بررسی طول‌ها
print("بررسی طول لیست‌ها:")
for key, value in examples.items():
    print(f"{key}: {len(value)}")

assert len(set(len(v) for v in examples.values())) == 1
df_examples = pd.DataFrame(examples)
print(f"✅ دیتاست آموزشی: {len(df_examples)} مثال")

# -------- ایندکس FAISS --------
print("🏗️ ساخت ایندکس FAISS...")
example_embeddings = embedder.encode(df_examples['text'].tolist(), show_progress_bar=True)
embed_dim = example_embeddings.shape[1]

index = faiss.IndexHNSWFlat(embed_dim, 32)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 50

faiss.normalize_L2(example_embeddings)
index.add(example_embeddings.astype(np.float32))
print(f"✅ ایندکس HNSW با {index.ntotal} نمونه")

# -------- پرامپت اصولی و جامع --------
PROMPT = """
شما تحلیل‌گر متخصص متون فارسی هستید و باید هر متن را فقط با توجه به اصول زیر در یکی از سه کلاس زیر طبقه‌بندی کنید:

۱. «تنفرآمیز»:
وقتی متن حاوی تحقیر، تهدید، طرد، کلیشه‌سازی منفی به صورت جمعی، تمسخر یا دعوت به نفرت/خشونت علیه یک گروه (بر اساس قومیت، ملیت، مذهب، جنسیت، سیاست یا سن) باشد؛
- اگر این موارد به شکل "آشکار" باشد (فحاشی یا تعمیم صریح گروهی)، یا به شکل "ضمنی" باشد (کنایه، اشاره غیرمستقیم خیلی منفی، طنز تلخ یا آیرونی با نیت تحقیر/تهدید/تضعیف کل گروه)—در هر دو صورت باید "تنفرآمیز" برچسب بزنید.
- شرط کلیدی: فقط وقتی برچسب "تنفرآمیز" بدهید که زمینه یا نیت پیام حاکی از تحقیر یا ترویج نگاه منفی به کل گروه باشد؛ حتی اگر غیرمستقیم باشد.
- صرف ذکر نام گروه یا ملیت یا شوخی/روایت بی‌قصد منفی کافی نیست به قبل و بعد کلمات در جمله هم حتما دقت کن و تصمیم کلی بگیر
-  برای تعمیم دادن سختگیرانه بررسی کن و اطمینان پیدا کن تیکه تنفر تعمیم داده شده باشه بعد برچسب هیت بزن
- واژه‌های تک تکشون و هرچی و همه و ها برای تعمیمه
۲. «توهین‌آمیز»:
اگر مخاطب فقط یک فرد واحد باشد و متن شامل توهین، تحقیر، تمسخر یا فحاشی به او باشد ولی هیچ تعمیم یا تحقیر گروهی وجود نداشته باشد.
- حتی اگر بسیار رکیک باشد، ولی علیه یک فرد یا جمع کاملاً مشخص (نه یک گروه اجتماعی)، فقط "توهین‌آمیز" است.
۳. «عادی»:
اگر متن فاقد تحقیر/کلیشه/تهدید جمعی باشد و صرفاً شامل روایت، گزارش، شوخی مثبت یا خنثی و منفی کم، گفت‌وگو، تعریف، نقد محترمانه، یا حتی بیان نام گروه بدون قصد منفی باشد.

◆ اگر مطمئن نبودی و متن مرزی یا با زمینه دوپهلو بود، به نیت و زمینه واقعی جمله توجه کن:
- اگر شواهدی از اراده تحقیر/تهدید گروه پیدا شد—even if embedded or subtle—آن را "تنفرآمیز" در نظر بگیر.
- ولی اگر فقط از نام بردن، روایت یا شوخی رایج استفاده شده و نشانه‌ای از تحقیر یا کلیشه‌سازی نیست، "عادی" زدن را ترجیح بده.

◇  هیچ‌گاه فقط به یک واژهٔ منفی توجه نکن؛ مهم این است که آیا هدف اصلی حمله به یک کل گروه است یا نه به برای موارد غیر مستقیم  ضمنی خیلی حواست باشه که تمایل به اینه که تنفرآمیز برچسب نزنیم .
-   اگر فعل باشه دیگه تنفر نیست ترک کردن هم فعل است و کرد هم میتونه فعل باشه به مواردی که یک کلمه چند معنی داره توجه کن و اطمینان حاصل کن که حتما معنی درستش رو برداشت کنی مثلا کرد هم فعل است هم قوم کرد میتونه باشه ولی جایی شک داشتی که فعل است یا یا نه فعل در نظر بگیر
- فقط یکی از طبقات (تنفرآمیز/توهین‌آمیز/عادی) را انتخاب کن و همیشه دلیل دقیق بر اساس این اصول بنویس.

خروجی شما باید فقط به صورت یک شیء JSON با فیلدهای زیر باشد:
{"label": "", "hate_type": "", "reason": ""}

"""

# -------- توابع (مشابه قبلی) --------
def get_similar_examples(text, top_k=6):
    qemb = embedder.encode([text], normalize_embeddings=True)
    D, I = index.search(qemb.astype(np.float32), top_k)

    results = []
    for idx in I[0]:
        if idx < len(df_examples):
            row = df_examples.iloc[idx]
            results.append({
                'text': row['text'],
                'label': row['label'],
                'hate_type': row['hate_type'],
                'reason': row['reason']
            })
    return results

def build_enhanced_prompt(text, examples):
    diverse_examples = []
    label_counts = Counter([ex['label'] for ex in examples])

    for label in ['تنفرآمیز', 'توهین‌آمیز', 'عادی']:
        label_examples = [ex for ex in examples if ex['label'] == label]
        diverse_examples.extend(label_examples[:2])

    ex_list = []
    for i, ex in enumerate(diverse_examples):
        ex_list.append(f"""مثال {i+1}:
متن: «{ex['text']}»
برچسب: {ex['label']}
نوع: {ex['hate_type']}
دلیل: {ex['reason']}""")

    ex_block = "\n\n".join(ex_list)

    prompt = f"""{PROMPT}

## نمونه‌های مرتبط:
{ex_block}

---
## متن تحلیل: «{text}»

تحلیل بر اساس اصول فوق:"""

    return prompt

def call_openrouter_api_enhanced(prompt, max_retries=5):
    data = {
        "model": "deepseek/deepseek-chat-v3-0324",
        "messages": [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 90,
        "temperature": 0.01,
        "top_p": 0.7
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(API_URL, headers=HEADERS, json=data, timeout=80)
            if response.status_code == 429:
                wait_time = min(5 * (2 ** attempt), 60)
                print(f"⏳ محدودیت نرخ - صبر {wait_time} ثانیه...")
                time.sleep(wait_time)
                continue
            response.raise_for_status()
            content = response.json()["choices"][0]["message"]["content"].strip()
            return content
        except Exception as e:
            print(f"❗ خطا تلاش {attempt+1}: {e}")
            time.sleep(3 * (attempt + 1))
    return None

def extract_json_enhanced(raw_response):
    try:
        cleaned_response = re.sub(r'``````', '', raw_response)

        json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', cleaned_response)
        if json_match:
            json_str = json_match.group()
            parsed = json.loads(json_str)

            if all(k in parsed for k in ["label", "hate_type", "reason"]):
                if parsed["label"] not in ["تنفرآمیز", "توهین‌آمیز", "عادی"]:
                    parsed["label"] = "نامشخص"
                return parsed

        label_patterns = {
            r'تنفرآمیز|نفرت\s*آمیز|hate': 'تنفرآمیز',
            r'توهین\s*آمیز|توهین|insult': 'توهین‌آمیز',
            r'عادی|normal|neutral': 'عادی'
        }

        label = "نامشخص"
        for pattern, detected_label in label_patterns.items():
            if re.search(pattern, raw_response, re.IGNORECASE):
                label = detected_label
                break

        hate_type = "N/A"
        hate_priorities = [
            (r'قومی|ethnic', 'قومی'),
            (r'ملیتی|national|ایران', 'ملیتی'),
            (r'مذهبی|religious', 'مذهبی'),
            (r'جنسیتی|gender', 'جنسیتی'),
            (r'سیاسی|political', 'سیاسی'),
            (r'سنی|age|ageism', 'سنی')
        ]

        for pattern, detected_type in hate_priorities:
            if re.search(pattern, raw_response, re.IGNORECASE):
                hate_type = detected_type
                break

        return {
            "label": label,
            "hate_type": hate_type,
            "reason": "تحلیل خودکار از پاسخ مدل"
        }

    except Exception as e:
        return {
            "label": "نامشخص",
            "hate_type": "N/A",
            "reason": f"خطا در تجزیه: {str(e)}"
        }

def classify_text_advanced(text, max_retries=3):
    for attempt in range(max_retries):
        try:
            examples = get_similar_examples(text, top_k=6)
            prompt = build_enhanced_prompt(text, examples)
            raw_response = call_openrouter_api_enhanced(prompt)

            if raw_response:
                result = extract_json_enhanced(raw_response)

                if (result['label'] != "نامشخص" and
                    len(result.get("reason", "")) > 8 and
                    result['reason'] != "تحلیل خودکار از پاسخ مدل"):
                    return result

            print(f"🔁 تلاش مجدد {attempt+1}/{max_retries} برای: [{text[:35]}...]")
            time.sleep(2 * (attempt + 1))

        except Exception as e:
            print(f"❗ خطا در تلاش {attempt+1}: {e}")
            time.sleep(3)

    return {
        "label": "نامشخص",
        "hate_type": "N/A",
        "reason": "عدم دریافت پاسخ معتبر پس از چندین تلاش"
    }

def process_file_enhanced(input_path, output_path=None, start_idx=0, end_idx=None, batch_size=30):
    if not os.path.isfile(input_path):
        print(f"❌ فایل {input_path} یافت نشد!")
        return None

    df = pd.read_csv(input_path)

    if 'Text' not in df.columns:
        print("❌ ستون 'Text' در فایل وجود ندارد!")
        return None

    if end_idx is None or end_idx > len(df):
        end_idx = len(df)
    if start_idx < 0:
        start_idx = 0
    if start_idx >= end_idx:
        print("❌ بازه نامعتبر")
        return None

    subset = df.iloc[start_idx:end_idx].copy()
    print(f"⏳ پردازش {len(subset)} ردیف با batch_size={batch_size}")
    print("📋 ویژگی‌های سیستم اصولی:")
    print("   🔬 پرامپت اصولی و جامع")
    print("   🎯 تمرکز بر نیت و زمینه")
    print("   ⚖️ اصول علمی تشخیص")
    print("   🔴 قومی/ملیتی: حساسیت بالا")
    print("   🟡 سایر موارد: حساسیت معمولی")

    results = []
    start_time = time.time()

    for i in range(0, len(subset), batch_size):
        batch = subset.iloc[i:i+batch_size]
        batch_results = []

        print(f"📦 پردازش batch {i//batch_size + 1}: ردیف‌های {start_idx+i} تا {start_idx+i+len(batch)-1}")

        for idx, row in batch.iterrows():
            text = str(row['Text'])[:400]
            result = classify_text_advanced(text)
            batch_results.append(result)
            # time.sleep(2.0)

        results.extend(batch_results)

        processed = len(results)
        total = len(subset)
        print(f"✅ تکمیل batch: {processed}/{total} ({processed/total*100:.1f}%)")

        if i + batch_size < len(subset):
            time.sleep(0)

    subset['Classification'] = [r['label'] for r in results]
    subset['Primary_Hate_Type'] = [r['hate_type'] for r in results]
    subset['Reason'] = [r['reason'] for r in results]

    if output_path is None:
        base, ext = os.path.splitext(input_path)
        output_path = f"{base}_principled_{start_idx}to{end_idx-1}.csv"

    subset.to_csv(output_path, index=False, encoding='utf-8-sig')

    elapsed = time.time() - start_time
    print(f"\n✅ تکمیل پردازش!")
    print(f"📁 فایل خروجی: {output_path}")
    print(f"⏱️ زمان کل: {elapsed:.1f} ثانیه")

    print(f"\n📊 آمار نهایی:")
    counts = subset['Classification'].value_counts()
    for label, count in counts.items():
        emoji = {"تنفرآمیز": "🔴", "توهین‌آمیز": "🟡", "عادی": "🟢", "نامشخص": "⚫"}.get(label, "⚪")
        percentage = count/len(subset)*100
        print(f"{emoji} {label}: {count} ردیف ({percentage:.1f}%)")

    if 'تنفرآمیز' in counts:
        hate_types = subset[subset['Classification'] == 'تنفرآمیز']['Primary_Hate_Type'].value_counts()
        print(f"\n🎯 انواع نفرت اصلی:")
        emoji_map = {"قومی": "🌍", "ملیتی": "🏳️", "مذهبی": "⛪", "جنسیتی": "⚧️", "سیاسی": "🏛️", "سنی": "👥"}
        for hate_type, count in hate_types.items():
            emoji = emoji_map.get(hate_type, "•")
            percentage = count/sum(hate_types)*100
            print(f"   {emoji} {hate_type}: {count} مورد ({percentage:.1f}%)")

    return output_path

# ======= تست سیستم اصولی =======
if __name__ == "__main__":

    output_path = process_file_enhanced('/content/out.csv', start_idx=14000, end_idx=18000)
    from google.colab import files
    files.download(output_path)


In [ ]:
import pandas as pd
from google.colab import files
import io

def upload_and_merge_csv_utf8():
    """
    تابعی برای دریافت و ادغام سه فایل CSV از کاربر با ذخیره در فرمت UTF-8 با BOM
    """

    print("لطفا سه فایل CSV خود را آپلود کنید:")
    uploaded = files.upload()

    # بررسی تعداد فایل‌های آپلود شده
    if len(uploaded) < 3:
        print(f"شما {len(uploaded)} فایل آپلود کردید. لطفا حداقل 3 فایل CSV آپلود کنید.")
        return None

    # خواندن فایل‌ها و تبدیل به DataFrame
    dataframes = []
    for filename in uploaded.keys():
        try:
            # تلاش برای خواندن با encoding های مختلف
            try:
                df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8')
            except:
                try:
                    df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8-sig')
                except:
                    df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='cp1256')

            print(f"فایل {filename} با موفقیت خوانده شد. تعداد ردیف‌ها: {len(df)}, تعداد ستون‌ها: {len(df.columns)}")
            dataframes.append(df)
        except Exception as e:
            print(f"خطا در خواندن فایل {filename}: {e}")
            return None

    # ادغام فایل‌ها با استفاده از outer join
    merged_df = pd.concat(dataframes, axis=0, join='outer', ignore_index=True)

    print(f"\nادغام با موفقیت انجام شد!")
    print(f"تعداد کل ردیف‌ها: {len(merged_df)}")
    print(f"تعداد کل ستون‌ها: {len(merged_df.columns)}")
    print(f"نام ستون‌ها: {list(merged_df.columns)}")

    # نمایش چند ردیف اول
    print("\nچند ردیف اول از فایل ادغام شده:")
    display(merged_df.head(10))

    # ذخیره فایل ادغام شده با فرمت UTF-8 با BOM
    merged_filename = "merged_file_utf8.csv"
    merged_df.to_csv(merged_filename, index=False, encoding='utf-8-sig')

    print(f"\nفایل ادغام شده با نام '{merged_filename}' با فرمت UTF-8 BOM ذخیره شد.")
    print("این فرمت از متون فارسی و عربی بهتر پشتیبانی می‌کند.")

    # دانلود فایل ادغام شده
    files.download(merged_filename)

    return merged_df

# اجرای تابع
result = upload_and_merge_csv_utf8()


In [ ]:
# نصب پکیج‌های ضروری
!pip install arabic-reshaper python-bidi --quiet

import pandas as pd
from google.colab import files
import matplotlib.pyplot as plt
import matplotlib as mpl
import arabic_reshaper
from bidi.algorithm import get_display
import io

# تابع برای تصحیح متن عربی/فارسی
def fix_arabic_text(text):
    """تبدیل متن فارسی/عربی به فرمت قابل نمایش در matplotlib"""
    reshaped_text = arabic_reshaper.reshape(text)
    return get_display(reshaped_text)

# ایجاد نمودارهای زیبا با پشتیبانی کامل فونت عربی
def create_beautiful_charts(df):
    """ایجاد چهار نوع نمودار مختلف با فونت عربی صحیح"""

    # تنظیمات matplotlib
    mpl.rcParams['font.family'] = 'DejaVu Sans'
    mpl.rcParams['figure.figsize'] = (15, 12)

    # محاسبه توزیع کلاس‌ها
    class_counts = df['hate_class'].value_counts()

    # تصحیح برچسب‌ها برای نمایش صحیح
    fixed_labels = [fix_arabic_text(label) for label in class_counts.index]

    # ایجاد subplot برای چهار نمودار
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(fix_arabic_text('تحلیل توزیع کلاس‌بندی داده‌ها'), fontsize=18, y=0.95)

    # رنگ‌های زیبا و متنوع
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8']

    # نمودار 1: میله‌ای عمودی
    ax1 = axes[0, 0]
    bars1 = ax1.bar(fixed_labels, class_counts.values, color=colors[:len(class_counts)])
    ax1.set_title(fix_arabic_text('نمودار میله‌ای توزیع کلاس‌ها'), fontsize=14, pad=20)
    ax1.set_xlabel(fix_arabic_text('کلاس'), fontsize=12)
    ax1.set_ylabel(fix_arabic_text('تعداد'), fontsize=12)

    # نمایش مقادیر روی میله‌ها
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{int(height)}', ha='center', va='bottom', fontsize=11, weight='bold')

    # نمودار 2: دایره‌ای
    ax2 = axes[0, 1]
    wedges, texts, autotexts = ax2.pie(class_counts.values, labels=fixed_labels,
                                       autopct='%1.1f%%', colors=colors[:len(class_counts)],
                                       startangle=90)
    ax2.set_title(fix_arabic_text('نمودار دایره‌ای نسبت کلاس‌ها'), fontsize=14, pad=20)

    # نمودار 3: میله‌ای افقی
    ax3 = axes[1, 0]
    bars3 = ax3.barh(range(len(class_counts)), class_counts.values,
                     color=colors[:len(class_counts)])
    ax3.set_yticks(range(len(class_counts)))
    ax3.set_yticklabels(fixed_labels)
    ax3.set_xlabel(fix_arabic_text('تعداد'), fontsize=12)
    ax3.set_title(fix_arabic_text('نمودار میله‌ای افقی'), fontsize=14, pad=20)

    # نمایش مقادیر کنار میله‌ها
    for i, bar in enumerate(bars3):
        width = bar.get_width()
        ax3.text(width + 0.1, bar.get_y() + bar.get_height()/2.,
                f'{int(width)}', ha='left', va='center', fontsize=11, weight='bold')

    # نمودار 4: حلقه‌ای (Donut)
    ax4 = axes[1, 1]
    wedges4, texts4 = ax4.pie(class_counts.values, labels=fixed_labels,
                             colors=colors[:len(class_counts)],
                             wedgeprops=dict(width=0.4), startangle=90)

    # ایجاد دایره مرکزی برای حلقه
    centre_circle = plt.Circle((0,0), 0.60, fc='white')
    ax4.add_artist(centre_circle)
    ax4.set_title(fix_arabic_text('نمودار حلقه‌ای'), fontsize=14, pad=20)

    plt.tight_layout()
    plt.show()

    # ذخیره فایل با کیفیت بالا
    plt.savefig('arabic_charts.png', dpi=300, bbox_inches='tight')
    files.download('arabic_charts.png')

    return class_counts

# تابع اصلی بارگذاری و تحلیل
def load_and_create_charts():
    """بارگذاری فایل CSV و ایجاد نمودارها"""

    print("لطفا فایل CSV خود را آپلود کنید:")
    uploaded = files.upload()

    if not uploaded:
        print("هیچ فایلی آپلود نشد!")
        return

    # خواندن فایل با encoding های مختلف
    filename = list(uploaded.keys())[0]
    try:
        df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8-sig')
    except:
        try:
            df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8')
        except:
            df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='cp1256')

    print(f"✅ فایل بارگذاری شد. تعداد ردیف‌ها: {len(df)}")
    print(f"📋 ستون‌های موجود: {list(df.columns)}")

    # بررسی وجود ستون classification
    if 'classification' not in df.columns:
        print("❌ ستون 'classification' در فایل یافت نشد!")
        print(f"ستون‌های موجود: {list(df.columns)}")
        return

    # نمایش آمار اولیه
    print(f"\n📊 توزیع کلاس‌های موجود:")
    class_dist = df['classification'].value_counts()
    for cls, count in class_dist.items():
        print(f"   {cls}: {count}")

    # ایجاد نمودارها
    print("\n🎨 در حال ایجاد نمودارها...")
    result = create_beautiful_charts(df)

    print("✨ نمودارها با موفقیت ایجاد شدند!")
    return df, result

# اجرای نهایی
result = load_and_create_charts()
